<a href="https://colab.research.google.com/github/OscarPerilla/unicafam_Modelos_Analiticos_Indra/blob/main/Sesion3_ejercicio3_Integracion_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Integración de datos**

In [ ]:
# Importar librerías de análisis y StringIO para leer texto como archivo CSV en memoria
import pandas as pd
import numpy as np
import sqlite3
from io import StringIO

In [ ]:
# Definir datos base de clientes en formato CSV separado por punto y coma
csv_clientes = """id_cliente;nombre;ciudad;segmento
C001;Ana Ruiz;Bogotá;Oro
C002;Luis Peña;Medellín;Plata
C003;Marta Gil;Cali;Oro
C004;Juan Ríos;Bogotá;Bronce
C005;Sofía Mora;Barranquilla;Plata
C006;Carlos Puenes;Pasto;Bronce
c007;Patricia Mojica;Manizales;plata
C008;Juan Velez;Bucaramanga;Bronce"""

In [ ]:
# Cargar el dataset de ejemplo de California Housing incluido en Colab
df_california_housing = pd.read_csv("/content/sample_data/california_housing_train.csv")
df_california_housing.head(4)

In [ ]:
# Cargar la tabla de clientes desde el texto CSV en memoria
clientes = pd.read_csv(StringIO(csv_clientes), sep=";")
clientes

In [ ]:
# Integrar datos de pedidos desde estructura JSON simulando una respuesta de API
respuesta_api = [
    {"pedido": 101, "cliente": "C001", "items": 3,
     "pago": {"medio": "Tarjeta", "valor": 150000}},
    {"pedido": 102, "cliente": "C003", "items": 1,
     "pago": {"medio": "PSE", "valor": 80000}},
    {"pedido": 103, "cliente": "C001", "items": 2,
     "pago": {"medio": "Efectivo", "valor": 45000}},
    {"pedido": 104, "cliente": "C009", "items": 5,
     "pago": {"medio": "Tarjeta", "valor": 210000}},
]
pedidos = pd.json_normalize(respuesta_api)
print(pedidos)

# **Merge de datos**

In [ ]:
# Comparar unión INNER vs LEFT para identificar clientes sin compras
inner = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="inner")
left = pd.merge(clientes, pedidos, left_on="id_cliente",
                right_on="cliente", how="left")

print("INNER:", inner.shape[0], "filas")
print("LEFT :", left.shape[0], "filas")
sin_compra = left[left["pedido"].isna()]["nombre"].tolist()
print("Clientes sin compras:", sin_compra)

In [ ]:
# Validar coincidencias y registros huérfanos mediante OUTER JOIN con indicator=True
outer = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="outer", indicator=True)
print(outer["_merge"].value_counts())
huerfanos = outer[outer["_merge"] == "right_only"]
print("Pedidos sin cliente:", huerfanos["pedido"].tolist())

# **Consolidación de datos**

In [ ]:
# Generar ventas mensuales simuladas por sucursal (Enero, Febrero y Marzo)
np.random.seed(1)

def ventas_mes(mes):
    return pd.DataFrame({
        "mes": mes,
        "sucursal": ["Norte", "Sur", "Centro"],
        "ventas": np.random.randint(80, 200, 3) * 1e6
    })

ene = ventas_mes("Ene")
feb = ventas_mes("Feb")
mar = ventas_mes("Mar")

# Consolidar los tres meses en un solo DataFrame
consolidado = pd.concat([ene, feb, mar], ignore_index=True)
consolidado

In [ ]:
# Calcular métricas agregadas por sucursal (total, promedio y meses)
resumen = consolidado.groupby("sucursal").agg(
    total=("ventas", "sum"),
    promedio=("ventas", "mean"),
    meses=("mes", "count")
)
resumen = (resumen / [1e6, 1e6, 1]).round(1)
print(resumen.sort_values("total", ascending=False))

In [ ]:
# Transformar tabla de formato largo a ancho (pivot_table) y viceversa (melt)
ancho = consolidado.pivot_table(index="sucursal", columns="mes",
                                values="ventas") / 1e6
ancho = ancho[["Ene", "Feb", "Mar"]]
print(ancho)

largo = ancho.reset_index().melt(id_vars="sucursal",
                                 var_name="mes", value_name="ventas_M")
print(largo.head(4))

# **Integración SQL**

In [ ]:
# Consultar ventas por ciudad en SQLite en memoria usando LEFT JOIN
con = sqlite3.connect(":memory:")
clientes.to_sql("clientes", con, index=False)
pedidos.rename(columns={"pago.valor": "valor"})[
    ["pedido", "cliente", "valor"]].to_sql("pedidos", con, index=False)

query = """
SELECT c.ciudad, COUNT(p.pedido) AS pedidos, SUM(p.valor) AS ventas
FROM clientes c LEFT JOIN pedidos p ON c.id_cliente = p.cliente
GROUP BY c.ciudad ORDER BY ventas DESC
"""
print(pd.read_sql(query, con))

# **Ejercicios Prácticos - Sesión 3**

## **Ejercicio 1: Clientes inactivos**
Marketing quiere enviar una campaña a los clientes que nunca han comprado y revisar los pedidos con clientes inexistentes.

In [ ]:
# 1. Agregar 3 clientes nuevos (C009-C011) a la tabla clientes
nuevos_clientes = pd.DataFrame({
    "id_cliente": ["C009", "C010", "C011"],
    "nombre": ["Laura Gómez", "Andrés Castro", "Diana Pérez"],
    "ciudad": ["Bogotá", "Medellín", "Cali"],
    "segmento": ["Oro", "Bronce", "Plata"]
})
clientes_act = pd.concat([clientes, nuevos_clientes], ignore_index=True)

# Normalizar IDs a mayúsculas para evitar diferencias como 'c007' vs 'C007'
clientes_act["id_cliente_norm"] = clientes_act["id_cliente"].str.upper()
pedidos["cliente_norm"] = pedidos["cliente"].str.upper()

# 2. Realizar LEFT JOIN con pedidos y listar los clientes sin compras
left_ej1 = pd.merge(
    clientes_act,
    pedidos,
    left_on="id_cliente_norm",
    right_on="cliente_norm",
    how="left"
)
clientes_sin_compra = left_ej1[left_ej1["pedido"].isna()]["nombre"].tolist()
print("Clientes sin compras:", clientes_sin_compra)

# 3. Realizar OUTER JOIN con indicator=True y contar cada categoría de _merge
outer_ej1 = pd.merge(
    clientes_act,
    pedidos,
    left_on="id_cliente_norm",
    right_on="cliente_norm",
    how="outer",
    indicator=True
)
print("\nConteo por categoría en _merge:")
print(outer_ej1["_merge"].value_counts())

# 4. Generar lista final para marketing y reporte de pedidos sin cliente registrado
lista_marketing = outer_ej1[outer_ej1["_merge"] == "left_only"][
    ["id_cliente", "nombre", "ciudad", "segmento"]
].drop_duplicates()

pedidos_sin_cliente = outer_ej1[outer_ej1["_merge"] == "right_only"][
    ["pedido", "cliente", "pago.valor"]
]

print("\nLista final para campaña de marketing (clientes sin compras):")
print(lista_marketing)
print("\nPedidos con cliente inexistente para revisar:")
print(pedidos_sin_cliente)

## **Ejercicio 2: Consolidado trimestral**
Gerencia necesita el crecimiento de cada sucursal entre enero y marzo.

In [ ]:
# 1. Pasar la tabla consolidado a formato ancho (sucursales x meses) con pivot_table
trimestral_ancho = consolidado.pivot_table(
    index="sucursal",
    columns="mes",
    values="ventas",
    aggfunc="sum"
)[["Ene", "Feb", "Mar"]]

# 2. Calcular la columna crecimiento_% = (Mar - Ene) / Ene * 100
trimestral_ancho["crecimiento_%"] = (
    (trimestral_ancho["Mar"] - trimestral_ancho["Ene"]) / trimestral_ancho["Ene"]
) * 100

# 3. Ordenar de mayor a menor e indicar la sucursal líder
resultado_ej2 = trimestral_ancho.sort_values("crecimiento_%", ascending=False)
print(resultado_ej2.round(2))

sucursal_lider = resultado_ej2.index[0]
mayor_crecimiento = resultado_ej2.iloc[0]["crecimiento_%"]
print(f"\nSucursal líder en crecimiento (Ene -> Mar): {sucursal_lider} ({mayor_crecimiento:.2f}%)")

## **Ejercicio 3: SQL vs Pandas**
Responde la misma pregunta de dos formas: ¿cuál es el medio de pago con más ventas?

In [ ]:
# 1. Resolver con pandas usando groupby("pago.medio")
ventas_medio_pd = (
    pedidos.groupby("pago.medio", as_index=False)["pago.valor"]
    .sum()
    .rename(columns={"pago.medio": "medio_pago", "pago.valor": "total_ventas"})
    .sort_values("total_ventas", ascending=False)
)
print("Resultado con pandas:")
print(ventas_medio_pd)
print("Medio de pago con más ventas (pandas):", ventas_medio_pd.iloc[0]["medio_pago"])

# 2. Crear la tabla pagos en SQLite con to_sql
con_ej3 = sqlite3.connect(":memory:")
pagos_sql = pedidos.rename(columns={"pago.medio": "medio_pago", "pago.valor": "valor"})
pagos_sql[["pedido", "cliente", "medio_pago", "valor"]].to_sql("pagos", con_ej3, index=False)

# 3. Escribir y ejecutar la consulta SELECT ... GROUP BY ... ORDER BY
query_ej3 = """
SELECT medio_pago, SUM(valor) AS total_ventas
FROM pagos
GROUP BY medio_pago
ORDER BY total_ventas DESC
"""
ventas_medio_sql = pd.read_sql(query_ej3, con_ej3)
print("\nResultado con SQL:")
print(ventas_medio_sql)

# 4. Comparar si ambos métodos entregan exactamente el mismo resultado
mismo_resultado = ventas_medio_pd.reset_index(drop=True).equals(ventas_medio_sql.reset_index(drop=True))
print("\n¿Dan exactamente el mismo resultado?:", mismo_resultado)